In [ ]:
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mj_sim/humanoid')

import numpy as np
import mujoco
import mujoco.viewer
from scipy.spatial import ConvexHull
from dataclasses import dataclass

from core import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel
from utils import SimScheduler, Visualizer, DataLogger, plot_helpers
from control import (WholeBodyController, MotionPlanner, TaskSpaceController,
                     GaitScheduler, TrajOptimizer, CentroidalMPC)


%load_ext autoreload
%autoreload 2

URDF = '/home/frlab/mj_opt/models/g1/urdf/g1_29dof.urdf'
PKG  = '/home/frlab/mj_opt/models/g1'
MJCF = '/home/frlab/mj_opt/models/scenes/g1/scene_29dof.xml'

In [ ]:
PHASE_OFFSET  = np.array([0.0, 0.5])   # LEG_KEYS 순서: L_foot=0, R_foot=1
HEIGHT_SWING  = 0.02                   # swing foot peak height [m] — 제자리걸음용 낮게
LEG_KEYS      = ["L_foot", "R_foot"]
LEG_IDX       = {k: i for i, k in enumerate(LEG_KEYS)}

# ─── 동작 모드 ─────────────────────────────────────────────────────────────
# STANDING_MODE: True=양발 stance, False=gait 기반 swing/stance 토글
# USE_MPC      : True=MPC + WBTorqueGen 직결 / False=기존 WBC.compute() 1-step QP
#
# 검증 상태 (헤드리스 시뮬 결과):
#   • STANDING + USE_MPC=False : 3s 안정 (legacy 검증 경로, drift <0.5cm).
#   • STANDING + USE_MPC=True  : ~1s 까진 안정.  3s 시뮬에선 ankle moment 부재로
#                                 천천히 발산 (xy drift 100cm+).  MPC 가 3D force 만
#                                 출력 (per leg 3D, moment=0 패딩) → angular 제어
#                                 부족.  근본 해결은 6D wrench MPC 또는 hybrid
#                                 (MPC=plan, ProxQP=force distribution).
#   • USE_MPC=True + 보행      : 단일 stance 200ms 너무 짧음 + ankle moment 부재로
#                                 즉시 발산.  보행은 추가 설계 필요 (gait timing,
#                                 swing/CoM 동기, capture point).
STANDING_MODE = True
USE_MPC       = False

# ─── MPC 설정 ──────────────────────────────────────────────────────────────
MPC_N        = 10           # horizon step 수
MPC_DT       = 0.05         # MPC step 간격 [s]  → horizon = 0.5s
MPC_INTERVAL = 5            # control step마다 MPC 호출 주기 (500Hz / 5 = 100Hz)

# state x = [Θ(3), p(3), ω(3), v(3), 1] — TrajOptimizer 컨벤션 (ω = world frame)
MPC_Q_DIAG = np.array([
    10, 20, 1,        # Θ: roll, pitch, yaw
     1,  1, 50,       # p:  px, py, pz (world)
     1,  1, 1,        # ω:  world (go2-convex-mpc 컨벤션)
     2,  2, 1,        # v:  vx, vy, vz (world)
     0               # augmented state (1) — Q 페널티 없음
], dtype=float)
# R 스케일 주의: u_z ≈ mg/n_legs ≈ O(170N).  R·u² 가 hover 비용,
# Q[pz]·(0.5·g·t²)² 가 추락 비용.  R 너무 크면 MPC 가 추락을 선호하게 됨.
# go2-convex-mpc 권장 범위 1e-7 ~ 1e-5.
MPC_R_DIAG = np.array([1e-5] * 6)   # 6 = 2 legs × 3D force

In [ ]:
class RobotOrchestrator:
    """USE_MPC 분기에 따라 균형 토크를 두 가지 방식 중 하나로 계산.
      USE_MPC=True  : Hybrid — MPC.solve() → ddc_feedforward → WBC.compute()
                      MPC 가 horizon 기반 desired CoM ddc 를 plan,
                      legacy PD 출력에 feedforward 로 합산.  PD 가 고대역 안정성 담당,
                      MPC 가 horizon-aware 보정 추가.  ProxQP 가 6D wrench 분배.
      USE_MPC=False : 순수 WBC.compute() — PD on COM_POS_DES (g1_stand 검증된 경로).
    swing leg는 두 경우 모두 Bezier + null-space task torque.
    """

    def __init__(self, kernel, wrapper, state, controller, planner,
                 gait, wbc, mpc, mpc_interval=MPC_INTERVAL, logger=None):
        self.kernel       = kernel
        self.wrapper      = wrapper
        self.state        = state
        self.controller   = controller
        self.planner      = planner
        self.gait         = gait
        self.wbc          = wbc
        self.mpc          = mpc
        self.mpc_interval = mpc_interval
        self.logger       = logger

        self.swing_pos_des   = {leg: np.zeros(3) for leg in LEG_KEYS}
        self.swing_pos_start = {
            leg: wrapper.get_ee_state_world(leg)[0].translation.copy()
            for leg in LEG_KEYS
        }
        self._prev_contact = np.ones(len(LEG_KEYS), dtype=int)

        # Hybrid: MPC 가 plan 한 desired CoM acceleration (3,).  None = 미계산 / PD 만.
        self._ddc_mpc      = None
        self._F_hat_wrench = np.zeros(12)
        self._step_counter = 0
        self._warned_mpc   = False
        self._gravity_vec  = np.array([0.0, 0.0, 9.81])


    # ─── x0, x_ref, contact_table 빌더 ────────────────────────────────────
    def _build_x0(self):
        """현재 augmented state (13,) — TrajOptimizer 컨벤션.
        state.humanoid_base_ang_vel 는 body frame (Pin freeflyer 규약) → R_bw 로 world 변환.
        """
        rpy   = self.state.compute_rpy_world()         # continuous yaw
        R_bw  = self.wrapper.R_body_to_world
        x0 = np.zeros(13)
        x0[0:3]  = rpy
        x0[3:6]  = self.wrapper.com_pos_world
        x0[6:9]  = R_bw @ self.state.humanoid_base_ang_vel    # body → world
        x0[9:12] = self.wrapper.com_vel_world          # world frame
        x0[12]   = 1.0
        return x0

    def _build_x_ref(self, current_yaw, t_now):
        """Reference horizon (N, 13).
        STANDING_MODE: CoM xy 는 COM_POS_DES 로 고정.
        보행 모드     : 매 horizon step k 의 stance foot xy 평균을 CoM xy ref 로 (LIPM 시프트).
                       제자리걸음 — 발 위치 변화 없다고 가정해 현재 위치 캐싱 후 horizon 전체에 사용.
        """
        N, dt = self.mpc.N, self.mpc.dt
        x_ref = np.zeros((N, 13))
        x_ref[:, 2]  = current_yaw
        x_ref[:, 5]  = COM_POS_DES[2]      # z 고정
        x_ref[:, 12] = 1.0

        if STANDING_MODE:
            x_ref[:, 3] = COM_POS_DES[0]
            x_ref[:, 4] = COM_POS_DES[1]
            return x_ref

        foot_xy = np.array([
            self.wrapper.get_ee_state_world(leg)[0].translation[:2]
            for leg in LEG_KEYS
        ])
        for k in range(N):
            ct_k, _ = self.gait.compute_gait_state(t_now + k * dt)
            stance_idx = np.where(ct_k == 1)[0]
            if len(stance_idx) == 0:
                ref_xy = foot_xy.mean(axis=0)
            else:
                ref_xy = foot_xy[stance_idx].mean(axis=0)
            x_ref[k, 3:5] = ref_xy
        return x_ref

    def _build_contact_table(self, t_now):
        """STANDING_MODE 면 horizon 전체 all-stance 강제 (gait 미래 swing 패턴이
        force 계획을 비대칭화하는 것 방지)."""
        N, dt = self.mpc.N, self.mpc.dt
        if STANDING_MODE:
            return np.ones((len(LEG_KEYS), N), dtype=int)
        ct = np.zeros((len(LEG_KEYS), N), dtype=int)
        for k in range(N):
            mask, _ = self.gait.compute_gait_state(t_now + k * dt)
            ct[:, k] = mask
        return ct


    # ─── 메인 콜백 ─────────────────────────────────────────────────────────
    def on_control(self, t):
        # 1) 상태 갱신
        self.kernel.update_robot_state(self.state)
        self.wrapper.update_model(self.state.q, self.state.dq)

        # 2) gait + swing 경로 생성
        if STANDING_MODE:
            contact_mask  = np.ones(len(LEG_KEYS), dtype=int)
            tau_swing     = np.zeros(self.wrapper.na)
        else:
            contact_mask, swing_phases = self.gait.compute_gait_state(t)

            for i, leg in enumerate(LEG_KEYS):
                if self._prev_contact[i] == 1 and contact_mask[i] == 0:
                    oMf, _, _ = self.wrapper.get_ee_state_world(leg)
                    self.swing_pos_start[leg] = oMf.translation.copy()
            self._prev_contact[:] = contact_mask

            J_c_stance = self._get_stance_jacobian(contact_mask)

            tau_swing = np.zeros(self.wrapper.na)
            for i, leg in enumerate(LEG_KEYS):
                if contact_mask[i] == 0:
                    pos_target = self.planner.compute_raibert_heuristic(leg=leg)
                    pos_des, vel_des, _ = self.planner.compute_bezier_trajectory(
                        swing_phases[i], self.swing_pos_start[leg],
                        pos_target, self.gait.swing_time, HEIGHT_SWING)
                    self.swing_pos_des[leg] = pos_des
                    tau_swing += self._swing_task_torque(leg, pos_des, vel_des, J_c_stance)

        # 3) 균형 토크: USE_MPC 분기
        if USE_MPC:
            # 3a) MPC 풀이 (mpc_interval 마다) → ddc_mpc 캐싱
            if self._step_counter % self.mpc_interval == 0:
                x0    = self._build_x0()
                x_ref = self._build_x_ref(current_yaw=x0[2], t_now=t)
                ct    = self._build_contact_table(t)
                try:
                    F_seq  = self.mpc.solve(x0, x_ref, ct)
                    status = self.mpc.last_status
                    if status in ('solved', 'solved inaccurate'):
                        # SRBD 1단계 force 합 → m·ddc = ΣF − mg ẑ
                        F_total = F_seq[0][0:3] + F_seq[0][3:6]
                        self._ddc_mpc = F_total / self.wrapper.mass - self._gravity_vec
                    elif not self._warned_mpc:
                        print(f"[MPC bad status @ t={t:.3f}] {status} — keep last ddc_mpc")
                        self._warned_mpc = True
                except RuntimeError:
                    if not self._warned_mpc:
                        print(f"[MPC fail @ t={t:.3f}] OSQP solver failed — keep last ddc_mpc")
                        self._warned_mpc = True

            # 3b) Hybrid: PD + MPC feedforward.  ProxQP 가 6D wrench 분배.
            tau_balance = self.wbc.compute(
                COM_POS_DES, np.zeros(3),
                contact_mask=contact_mask,
                ddc_feedforward=self._ddc_mpc,
            )
            self._F_hat_wrench = self.wbc.force_opt.opt_F.copy()
        else:
            # 순수 legacy WBC (PD only)
            tau_balance = self.wbc.compute(COM_POS_DES, np.zeros(3), contact_mask=contact_mask)
            self._F_hat_wrench = self.wbc.force_opt.opt_F.copy()
        self._step_counter += 1

        tau = tau_balance + tau_swing
        self.kernel.ctrl_tau = tau

        # 4) 로깅
        if self.logger is not None:
            F_hat = self._F_hat_wrench
            self.logger.log(
                t            = t,
                com_des      = COM_POS_DES,
                com_curr     = self.wrapper.com_pos_world,
                com_vel      = self.wrapper.com_vel_world,
                contact_mask = contact_mask,
                F_L          = F_hat[0:3].copy(),
                F_R          = F_hat[6:9].copy(),
                tau          = tau,
                q_act        = self.state.q[7:],
                dq_act       = self.state.dq[6:],
            )


    def _get_stance_jacobian(self, contact_mask):
        Js = [self.wrapper.get_J_world(leg)
              for i, leg in enumerate(LEG_KEYS) if contact_mask[i] == 1]
        return np.vstack(Js) if Js else None

    def _swing_task_torque(self, leg, pos_des, vel_des, J_c_stance):
        """swing 발 위치 추종 → null-space projected tau (na,)."""
        oMf, lin_vel, _ = self.wrapper.get_ee_state_world(leg)
        pos_err = pos_des - oMf.translation
        vel_err = vel_des - lin_vel

        J_p = self.wrapper.get_J_world(leg)[:3]
        Kp, Kd = self.controller.Kp_pos, self.controller.Kd_lin
        F_des  = Kp @ pos_err + Kd @ vel_err

        M_inv   = self.wrapper.M_inv
        JMinvJT = J_p @ M_inv @ J_p.T + self.controller.damping2 * np.eye(3)
        tau_pure = J_p.T @ np.linalg.solve(JMinvJT, F_des)

        if J_c_stance is not None:
            nc          = J_c_stance.shape[0]
            M_inv_tau   = M_inv @ tau_pure
            JcMinvJcT   = J_c_stance @ M_inv @ J_c_stance.T
            gamma       = np.linalg.solve(JcMinvJcT + 1e-6 * np.eye(nc),
                                          J_c_stance @ M_inv_tau)
            tau_pure    = tau_pure - J_c_stance.T @ gamma

        return tau_pure[6:]

    def on_render(self, t, visualizer):
        _, points = self.kernel.get_foot_contact_state()
        if len(points) >= 3:
            pts = np.array(points)
            try:
                hull  = ConvexHull(pts[:, :2])
                z     = pts[:, 2].mean()
                verts = pts[hull.vertices]
                n     = len(verts)
                for j in range(n):
                    p1 = np.array([verts[j][0],       verts[j][1],       z])
                    p2 = np.array([verts[(j+1)%n][0], verts[(j+1)%n][1], z])
                    visualizer.draw_line(p1, p2, rgba=(0, 1, 0, 1), thickness=0.005)
            except Exception:
                pass
        for leg in LEG_KEYS:
            visualizer.draw_trajectory(self.swing_pos_des[leg], rgba=(1.0, 0.5, 0.0, 1.0))

상태 초기화, wrapper 호출, 커널 호출, 제어기 호출, 초기 ee SE3, SO3, x, y, z 추출 (피노키오 컨벤션)

In [ ]:
state    = FloatingBaseRobotState()
wrapper  = Pinocchio_Wrapper(URDF, PKG)
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
kernel   = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)
kernel.register_foot_geoms({"L_foot": "left_ankle_roll_link",
                             "R_foot": "right_ankle_roll_link"})

controller = TaskSpaceController(wrapper)
gait       = GaitScheduler(gait_hz=0.5, duty=0.9, phase_offset=PHASE_OFFSET)
optimizer  = TrajOptimizer(wrapper)
planner    = MotionPlanner(wrapper, optimizer, gait)
wbc        = WholeBodyController(wrapper)

# 키프레임에서 Pin 포맷으로 정확히 변환해서 기준값 추출
kernel.reset_to_keyframe()
kernel.update_robot_state(state)
wrapper.update_model(state.q, np.zeros(wrapper.nv))

COM_HEIGHT_DES = float(wrapper.com_pos_world[2])
COM_POS_DES    = wrapper.com_pos_world.copy()   # 수평 기준도 고정
print(f"COM_POS_DES = {COM_POS_DES}")

# Raibert: 제자리걸음 목표 — 현재 CoM xy 유지, 속도/yaw_rate=0
optimizer.set_reference(pos_x=COM_POS_DES[0], pos_y=COM_POS_DES[1],
                        vel_x=0.0, vel_y=0.0, yaw_rate=0.0)

# fz_min 동적 산정 — mg / (5 · n_legs).  제자리걸음 시 한 발이 지나치게 언로드되는 해를 막음.
mg          = wrapper.mass * 9.81
n_legs      = len(LEG_KEYS)
FZ_MIN      = mg / (5.0 * n_legs)
FZ_MAX      = max(500.0, 2.0 * mg)            # mg의 2배까진 허용
print(f"mass={wrapper.mass:.2f}kg, mg={mg:.1f}N, fz_min={FZ_MIN:.1f}N/leg, fz_max={FZ_MAX:.1f}N/leg")

# MPC 인스턴스 (TrajOptimizer 공유 — 동일 wrapper로 dynamics 갱신)
mpc = CentroidalMPC(optimizer, N=MPC_N, dt=MPC_DT,
                    Q_diag=MPC_Q_DIAG, R_diag=MPC_R_DIAG,
                    mu=0.5, fz_min=FZ_MIN, fz_max=FZ_MAX)
print(f"✅ MPC init: N={mpc.N}, dt={mpc.dt}, nvars={mpc.nvars}, NU={mpc.NU}")

# Warm-up solve (sparsity & cache 활성화)
x0_warm = np.zeros(13)
x0_warm[3:6] = COM_POS_DES
x0_warm[12]  = 1.0
ct_warm = np.ones((len(LEG_KEYS), MPC_N), dtype=int)
_ = mpc.solve(x0_warm, np.tile(x0_warm, (MPC_N, 1)), ct_warm)
print(f"   warm-up: solve {mpc.solve_time:.2f}ms, status={mpc.last_status}")

init_oMb      = wrapper.oMb
init_oM_Lfoot = wrapper.oM_Lfoot
init_oM_Rfoot = wrapper.oM_Rfoot
init_oM_Lhand = wrapper.oM_Lhand
init_oM_Rhand = wrapper.oM_Rhand

Main Loop (상태 갱신, 제어, 뷰어, 시각화)

In [ ]:
# 실행 직전 초기화
kernel.reset_to_keyframe()
state.reset_yaw_unwrap()
mpc.x_prev = mpc.lam_x_prev = mpc.lam_a_prev = None  # warm start 리셋
logger = DataLogger()

# 오케스트레이터 + 스케줄러 실행
orch  = RobotOrchestrator(kernel, wrapper, state, controller, planner,
                          gait, wbc, mpc, mpc_interval=MPC_INTERVAL, logger=logger)
sched = SimScheduler(kernel.model, kernel.data, ctrl_hz=500, render_hz=60)
sched.run(on_control=orch.on_control, on_render=orch.on_render)

In [ ]:
import matplotlib.pyplot as plt

# 로그 추출
times    = logger.get('t')
com_des  = logger.get('com_des')         # (N, 3)
com_curr = logger.get('com_curr')        # (N, 3)
com_vel  = logger.get('com_vel')         # (N, 3)
F_L      = logger.get('F_L')             # (N, 3)
F_R      = logger.get('F_R')             # (N, 3)
tau      = logger.get('tau')             # (N, na)
mask     = logger.get('contact_mask')    # (N, n_legs)
q_act    = logger.get('q_act')           # (N, na)
dq_act   = logger.get('dq_act')          # (N, na)

mass_g = wrapper.mass * 9.81
print(f"총 step={len(times)}, 모드={'STANDING' if STANDING_MODE else 'WALKING'}, mg={mass_g:.1f}N")

# 1. CoM 추종 (XYZ 오차)
plot_helpers.plot_ee_tracking(times, com_des, com_curr,
                              title="CoM Position Tracking (des - curr)")

# 2. CoM 속도 (정지 상태에서 0 근방 유지 확인)
plot_helpers.plot_velocity_tracking(times, np.zeros_like(com_vel), com_vel,
                                    title="CoM Velocity (des=0)")

# 3. Contact force (fx, fy, fz; fz 합 ≈ mg 이어야 함)
#    plot_contact_schedule 시그니처: contact_mask (n_legs, N), force (3*n_legs, N)
contact_mat = mask.T                                                # (n_legs, N)
force_mat   = np.vstack([F_L.T, F_R.T])                              # (6, N)
dt = float(np.mean(np.diff(times))) if len(times) > 1 else 1/500
plot_helpers.plot_contact_schedule(times, contact_mat, force_mat, dt,
                                   leg_names=LEG_KEYS,
                                   title="Contact Forces (QP solution)")

# 4. fz 합 vs mg
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(times, F_L[:, 2], label='fz_L')
ax.plot(times, F_R[:, 2], label='fz_R')
ax.plot(times, F_L[:, 2] + F_R[:, 2], 'k-', linewidth=2, label='fz_L + fz_R')
ax.axhline(mass_g, color='r', linestyle='--', label=f'mg = {mass_g:.1f} N')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Vertical Force [N]')
ax.set_title('Vertical Contact Force vs Gravity'); ax.grid(True); ax.legend()
plt.tight_layout(); plt.show()

# 5. Joint torque
plot_helpers.plot_joint_state(times, tau, dq_act,
                              joint_names=joint_names,
                              title="Torque (top) & Joint Velocity (bottom)")